In [ ]:
!pip install langchain langchain-community langchain-huggingface langchain-groq langchain-chroma chromadb sentence-transformers

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import shutil, os
from google.colab import drive
drive.mount('/content/drive')

import json
from langchain_core.documents import Document

json_path = "/content/drive/MyDrive/bcorp_extracted.json"

with open(json_path) as f:
    raw = json.load(f)

documents = [
    Document(
        page_content=json.dumps(company, indent=2),
        metadata={
            "source_file": filename,
            "company_name": company.get("company_name", "Unknown"),
            "headquarters": company.get("headquarters", "Unknown"),
            "certified_since": company.get("certified_since", "Unknown"),
            "industry": company.get("industry", "Unknown"),
            "sector": company.get("sector", "Unknown"),
            "operates_in": ", ".join(company.get("operates_in", [])),
            "website": company.get("website", "Unknown"),
            "b_impact_score": float(company.get("b_impact_score") or 0.0),
            "governance_score": float(company.get("governance_score") or 0.0),
        }
    )
    for filename, company in raw.items()
]

print(f"Loaded {len(documents)} documents")

# build the vector store
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from chromadb.api.client import SharedSystemClient
import shutil, os

PERSIST_DIR = "./bcorp_chroma_db"

SharedSystemClient.clear_system_cache()

if os.path.exists(PERSIST_DIR):
    shutil.rmtree(PERSIST_DIR)

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    persist_directory=PERSIST_DIR,
    collection_name="bcorp_companies"
)

print(f"ChromaDB built with {vectorstore._collection.count()} vectors!")

# build the RAG chain (with chat memory)
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from google.colab import userdata

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    max_tokens=1024,
    api_key=userdata.get('groq')
)

def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)


condense_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Given the conversation history and a follow-up question, rewrite the follow-up "
     "into a standalone question that includes any necessary context from the history. "
     "If the follow-up is already standalone, return it unchanged. "
     "Only output the rewritten question, nothing else."),
    MessagesPlaceholder("chat_history"),
    ("human", "{question}"),
])

condense_chain = condense_prompt | llm | StrOutputParser()

def get_standalone_question(input_dict):
    if not input_dict["chat_history"]:
        return input_dict["question"]
    return condense_chain.invoke(input_dict)

def retrieve_and_format(input_dict):
    standalone_question = get_standalone_question(input_dict)
    docs = retriever.invoke(standalone_question)
    return format_docs(docs)

#  using retrieved context + chat history
answer_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an expert assistant on B Corp certified companies. "
     "Use ONLY the context below to answer the question. "
     "If the answer is not in the context, say I don't have that information.\n\n"
     "CONTEXT:\n{context}"),
    MessagesPlaceholder("chat_history"),
    ("human", "{question}"),
])

rag_chain = (
    RunnablePassthrough.assign(context=RunnableLambda(retrieve_and_format))
    | answer_prompt | llm | StrOutputParser()
)

print("RAG chain with memory ready!")

# FastAPI server wired to rag_chain
import threading

import nest_asyncio
import uvicorn
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from pyngrok import ngrok
from langchain_core.messages import HumanMessage, AIMessage

app = FastAPI()


MAX_HISTORY_MESSAGES = 10


app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)


class ChatRequest(BaseModel):
    # Full conversation: [{"role": "user"|"assistant", "content": "..."}]
    messages: list


def to_lc_messages(messages):
    lc_messages = []
    for m in messages:
        role = m.get("role")
        content = m.get("content", "")
        if role == "user":
            lc_messages.append(HumanMessage(content=content))
        elif role == "assistant":
            lc_messages.append(AIMessage(content=content))
    return lc_messages


@app.post("/chat")
def chat(req: ChatRequest):

    if not req.messages or req.messages[-1].get("role") != "user":
        return {"reply": "I didn't receive a question."}

    question = req.messages[-1].get("content", "")
    history = req.messages[:-1][-MAX_HISTORY_MESSAGES:]
    chat_history = to_lc_messages(history)

    try:
        answer = rag_chain.invoke({"question": question, "chat_history": chat_history})
    except Exception as e:
        return {"reply": f"Something went wrong on the backend: {e}"}

    return {"reply": answer}


@app.get("/health")
def health():
    return {"status": "ok"}


# Tunnel + serve
ngrok.set_auth_token(userdata.get('NGROK'))

import time


def _run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")


server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()
time.sleep(2)  # give uvicorn a moment to bind the port before opening the tunnel

public_url = ngrok.connect(8000)
print("=" * 60)
print("PASTE THIS INTO API_URL IN YOUR HTML:")
print(public_url.public_url)
print("=" * 60)